# TP 3 — Normaliser et encoder : préparer les données pour un modèle
**Analyse de données — Master IA / Master RSD** · Chapitre 2 : sections 6 (Normalisation) et 7 (Encodage)

*Mettre les variables à la même échelle, transformer le texte en nombres, sans introduire de relations artificielles.*

**Objectifs du TP**
1. **Identifier la nature** de chaque variable (quantitative, nominale, ordinale, cible).
2. **Normaliser** les variables numériques : Z-score, Min-Max, transformation logarithmique → *choisir la méthode adaptée*.
3. **Encoder** les variables catégorielles : One-Hot, Ordinal, Label → *éviter les pièges classiques*.
4. **Assembler** une préparation reproductible (`ColumnTransformer`) et **mesurer son effet** sur un modèle k-NN.
5. **Documenter** chaque décision dans ce notebook.

| Volume | Organisation | Données | Livrable |
|---|---|---|---|
| 2 séances de 2 h (4 h au total) | Binômes · Notebook Jupyter | `clients_enrichi.csv` (fictif, fourni) | Notebook complété + conclusions |

⭐ **Continuité :** `clients_enrichi.csv` est le fichier **nettoyé du TP 2** (988 clients, sans valeur manquante), enrichi de trois variables catégorielles : `ville`, `niveau_etude`, `satisfaction`.

**Mode d'emploi** : complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Auto-correction** qui suit. Répondez aux questions dans les cellules **✍️**.

---
## Mise en place

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

---
## Étape 1 — Identifier la nature de chaque variable
*La méthode dépend de la nature **et** du rôle de la variable* (section 7, bonne pratique ①).

### 1.0 Charger le fichier

In [ ]:
df = ...   # TODO : pd.read_csv(...) — séparateur ";"

print(df.shape)
df.head()

✅ **Auto-correction**

In [ ]:
assert df.shape == (988, 11), f"Attendu (988, 11), obtenu {df.shape} -> séparateur ?"
assert df.isna().sum().sum() == 0, "le fichier est déjà nettoyé : aucune valeur manquante"
print("✅ 1.0 correct")

### 1.1 Classer les variables
On veut **prédire le `segment`** du client (A, B ou C) à partir des autres variables.
Pour chaque colonne, indiquez sa nature parmi : `"identifiant"`, `"quantitative"`, `"nominale"`, `"ordinale"`, `"cible"`.

💡 Regardez les modalités avec `df["colonne"].unique()` en cas de doute.

In [ ]:
for col in ["ville", "niveau_etude", "satisfaction", "segment"]:
    print(f"{col:13s}", sorted(df[col].unique()))

In [ ]:
nature = {
    "id":           ...,   # TODO
    "age":          ...,
    "revenu":       ...,
    "nb_enfants":   ...,
    "anciennete":   ...,
    "nb_achats":    ...,
    "ville":        ...,
    "niveau_etude": ...,
    "satisfaction": ...,
    "email":        ...,
    "segment":      ...,
}

✅ **Auto-correction**

In [ ]:
assert nature["ville"] == "nominale", "y a-t-il un ordre naturel entre les villes ?"
assert nature["niveau_etude"] == nature["satisfaction"] == "ordinale"
assert nature["segment"] == "cible"
assert nature["id"] == nature["email"] == "identifiant"
assert all(nature[c] == "quantitative" for c in ["age", "revenu", "nb_enfants", "anciennete", "nb_achats"])
print("✅ 1.1 correct")

In [ ]:
# Ces listes servent dans tout le TP
num_cols = [c for c, t in nature.items() if t == "quantitative"]
nom_cols = [c for c, t in nature.items() if t == "nominale"]
ord_cols = [c for c, t in nature.items() if t == "ordinale"]
print("numériques :", num_cols)
print("nominales  :", nom_cols)
print("ordinales  :", ord_cols)

---
## Étape 2 — Normaliser les variables numériques (section 6)

### 2.1 Le problème : des échelles différentes
Affichez les statistiques descriptives des variables numériques. Comparez l'**étendue** de `age` et de `revenu`.

In [ ]:
# TODO : statistiques descriptives de num_cols

etendue_age = ...      # TODO : max - min
etendue_revenu = ...   # TODO
print(etendue_age, etendue_revenu, round(etendue_revenu / etendue_age))

✅ **Auto-correction**

In [ ]:
assert etendue_age == 69
assert etendue_revenu / etendue_age > 2000, "le revenu varie plus de 2000 fois plus que l'âge"
print("✅ 2.1 correct")

✍️ **Question :** dans un k-NN, la distance entre deux clients est $\sqrt{(\Delta \text{age})^2 + (\Delta \text{revenu})^2 + \dots}$. Quelle variable va **dominer** cette distance ? Pourquoi est-ce un problème ?

…

### 2.2 Méthode 1 — Standardisation (Z-score)
$$z = \frac{x - \mu}{\sigma}$$
1. Calculez `revenu_z` **à la main** avec la formule (utilisez `std(ddof=0)`, l'écart-type de la population, comme scikit-learn).
2. Calculez-le avec `StandardScaler` et vérifiez que vous obtenez la même chose.

In [ ]:
revenu_z = ...         # TODO : formule à la main

scaler = ...           # TODO : StandardScaler()
revenu_z_sk = ...      # TODO : fit_transform sur df[["revenu"]] (double crochet : un DataFrame) puis .ravel()

print(round(revenu_z.mean(), 6), round(revenu_z.std(ddof=0), 6))

✅ **Auto-correction**

In [ ]:
assert np.allclose(revenu_z, revenu_z_sk), "la formule et StandardScaler doivent donner le même résultat"
assert abs(revenu_z.mean()) < 1e-9 and abs(revenu_z.std(ddof=0) - 1) < 1e-9, "moyenne 0, écart-type 1"
assert np.isclose(scaler.mean_[0], df["revenu"].mean()), "le scaler mémorise μ"
print("✅ 2.2 correct")

### 2.3 Méthode 2 — Normalisation Min-Max
$$x' = \frac{x - x_{min}}{x_{max} - x_{min}} \in [0, 1]$$
Même exercice : à la main, puis avec `MinMaxScaler`.

In [ ]:
revenu_mm = ...        # TODO : formule à la main
revenu_mm_sk = ...     # TODO : MinMaxScaler().fit_transform(...).ravel()

✅ **Auto-correction**

In [ ]:
assert np.allclose(revenu_mm, revenu_mm_sk)
assert revenu_mm.min() == 0 and revenu_mm.max() == 1
print("✅ 2.3 correct")

### 2.4 Transformation logarithmique : corriger l'asymétrie
Le revenu est **asymétrique à droite** (vu au TP 2). Z-score et Min-Max **ne changent pas la forme** de la distribution ; le logarithme, si.

Calculez `revenu_log = np.log1p(revenu)` ($\log(1+x)$, défini même pour $x = 0$), puis comparez l'**asymétrie** (`skew()`) avant / après.

In [ ]:
revenu_log = ...   # TODO

skew = pd.Series({
    "brut":    ...,   # TODO : df["revenu"].skew()
    "Z-score": ...,
    "Min-Max": ...,
    "log":     ...,
}).round(2)
skew

✅ **Auto-correction**

In [ ]:
assert skew["brut"] == skew["Z-score"] == skew["Min-Max"], "Z-score et Min-Max conservent la forme"
assert abs(skew["log"]) < 0.3 < skew["brut"], "le log rend la distribution presque symétrique"
print("✅ 2.4 correct")

Cellule fournie : les quatre versions du revenu côte à côte (comme sur la diapositive « Comparer : Z-score vs Min-Max »).

In [ ]:
versions = {"Brut (DA)": df["revenu"], "Z-score": revenu_z, "Min-Max": revenu_mm, "log(1 + revenu)": revenu_log}
couleurs = ["gray", "tab:blue", "tab:green", "tab:purple"]

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, (titre, serie), c in zip(axes, versions.items(), couleurs):
    sns.histplot(serie, bins=30, kde=True, color=c, ax=ax)
    ax.set_title(f"{titre}\nskew = {serie.skew():.2f}", fontsize=10)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

### 2.5 Sensibilité aux valeurs extrêmes
Au TP 2, on a remplacé le revenu **999 999** (erreur de saisie) par une valeur imputée. Que se serait-il passé si on l'avait **oublié** ?

Sur une copie du revenu où l'on remet **une** valeur à 999 999, calculez la **médiane** des valeurs transformées par Min-Max et par Z-score, avec et sans l'outlier.

In [ ]:
avec_outlier = df["revenu"].copy()
avec_outlier.iloc[0] = 999999

def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

effet = pd.DataFrame({
    "sans outlier": ...,   # TODO : {"médiane Min-Max": ..., "médiane Z-score": ...} calculées sur df["revenu"]
    "avec outlier": ...,   # TODO : idem sur avec_outlier
}).round(3)
effet

✅ **Auto-correction**

In [ ]:
assert effet.loc["médiane Min-Max", "avec outlier"] < 0.05 < effet.loc["médiane Min-Max", "sans outlier"]
print("✅ 2.5 correct")

✍️ **Questions**
1. Que devient la plage occupée par les clients « normaux » après Min-Max quand l'outlier est présent ?
2. D'après la diapositive « Règle de choix », quelle méthode choisir pour le revenu ? Et pour `age` ?

1. …
2. …

### 2.6 Bonne pratique : apprendre μ et σ sur l'entraînement uniquement
*« Toujours calculer μ et σ sur l'ensemble d'entraînement uniquement. Appliquer la même transformation sur validation, test et nouvelles données. »*

1. Séparez `df` en entraînement (80 %) et test (20 %) avec `train_test_split(..., test_size=0.2, random_state=42, stratify=df["segment"])`.
2. Ajustez (`fit`) un `StandardScaler` sur `num_cols` du **train** seulement, puis transformez (`transform`) train **et** test.

In [ ]:
df_train, df_test = ...   # TODO

scaler = StandardScaler()
# TODO : fit sur le train
X_train_num = ...   # TODO : transform du train
X_test_num = ...    # TODO : transform du test (sans refaire fit !)

print("moyenne train :", X_train_num.mean(axis=0).round(3))
print("moyenne test  :", X_test_num.mean(axis=0).round(3))

✅ **Auto-correction**

In [ ]:
assert len(df_train) == 790 and len(df_test) == 198
assert np.allclose(X_train_num.mean(axis=0), 0), "le train est centré"
assert not np.allclose(X_test_num.mean(axis=0), 0), "le test ne doit PAS être exactement centré"
assert np.allclose(scaler.mean_, df_train[num_cols].mean()), "μ appris sur le train seulement"
print("✅ 2.6 correct")

✍️ **Question :** pourquoi les moyennes du **test** ne valent-elles pas exactement 0 ? Pourquoi ne faut-il **pas** refaire `fit` sur le test ?

…

---
## Étape 3 — Encoder les variables catégorielles (section 7)
Rappel du guide de décision : **ordre naturel ?** non → **One-Hot** ; oui → est-ce la **cible** ? oui → **Label**, non → **Ordinal**.

### 3.1 One-Hot Encoding — `ville` (nominale)
1. Encodez `ville` avec `pd.get_dummies(..., prefix="ville", dtype=int)`.
2. Refaites-le avec `drop_first=True`. Combien de colonnes dans chaque cas ?

In [ ]:
ville_oh = ...        # TODO
ville_oh_df = ...     # TODO : avec drop_first=True

print(ville_oh.shape, ville_oh_df.shape)
ville_oh.head()

✅ **Auto-correction**

In [ ]:
assert ville_oh.shape == (988, 5), "5 villes -> 5 colonnes binaires"
assert (ville_oh.sum(axis=1) == 1).all(), "exactement un 1 par ligne"
assert ville_oh_df.shape == (988, 4)
print("✅ 3.1 correct")

✍️ **Questions**
1. Avec `drop_first=True`, quelle ville a disparu ? Comment reconnaît-on encore un client de cette ville ?
2. Pourquoi serait-il **faux** d'encoder `ville` par Oran = 0, Alger = 1, Béchar = 2… ?

1. …
2. …

### 3.2 Ordinal Encoding — `niveau_etude` (ordinale)
1. Essayez d'abord `OrdinalEncoder()` **sans préciser l'ordre** (cellule fournie) : quel code reçoit chaque niveau ?
2. Corrigez en donnant l'ordre explicitement : `categories=[["Licence", "Master", "Doctorat"]]`.

In [ ]:
piege = OrdinalEncoder().fit(df[["niveau_etude"]])
print("Ordre appris par défaut :", list(piege.categories_[0]))

In [ ]:
enc_niveau = ...   # TODO : OrdinalEncoder avec l'ordre correct
niveau_code = ...  # TODO : fit_transform(...).ravel()

pd.crosstab(df["niveau_etude"], niveau_code)

✅ **Auto-correction**

In [ ]:
codes = pd.Series(niveau_code, index=df.index).groupby(df["niveau_etude"]).first()
assert codes["Licence"] < codes["Master"] < codes["Doctorat"], "l'ordre naturel doit être respecté"
assert df["revenu"].corr(pd.Series(niveau_code, index=df.index)) > 0.4, "niveau et revenu sont liés"
print("✅ 3.2 correct")

✍️ **Question :** que donnait l'encodage par défaut ? Pourquoi est-ce grave ici ?

…

### 3.3 Label Encoding — `segment` (cible)
Encodez la cible avec `LabelEncoder` et affichez le **mapping appris** (`le.classes_`).

In [ ]:
le = ...   # TODO : LabelEncoder()
y = ...    # TODO : fit_transform sur df["segment"]

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
mapping

✅ **Auto-correction**

In [ ]:
assert list(le.classes_) == ["A", "B", "C"] and set(y) == {0, 1, 2}
assert (le.inverse_transform(y) == df["segment"]).all(), "on doit pouvoir revenir aux étiquettes"
print("✅ 3.3 correct")

### 3.4 Piège : une catégorie inconnue au moment de la prédiction
Un nouveau client arrive de **Tlemcen**, une ville absente des données d'entraînement.
1. Cellule fournie : un `OneHotEncoder` par défaut **plante**.
2. Créez `enc_ville` avec `handle_unknown="ignore"` et `sparse_output=False`, entraînez-le sur `df_train[["ville"]]`, puis encodez le nouveau client.

In [ ]:
nouveau = pd.DataFrame({"ville": ["Tlemcen"]})

try:
    OneHotEncoder().fit(df_train[["ville"]]).transform(nouveau)
except ValueError as e:
    print("❌ Erreur :", str(e)[:90], "...")

In [ ]:
enc_ville = ...    # TODO
code_nouveau = ... # TODO : transform(nouveau)

print(enc_ville.get_feature_names_out(), code_nouveau)

✅ **Auto-correction**

In [ ]:
assert code_nouveau.shape == (1, 5) and code_nouveau.sum() == 0, "ville inconnue -> que des 0"
print("✅ 3.4 correct")

---
## Étape 4 — Assembler la préparation et mesurer son effet
Dans un vrai projet, on regroupe toutes les transformations dans un **`ColumnTransformer`** : chaque groupe de colonnes reçoit son traitement, et le même objet est appliqué au train et au test (bonne pratique ⑤ : *même encodage en entraînement et en test*).

### 4.1 Le `ColumnTransformer`
Complétez `prep` : `StandardScaler` sur `num_cols`, `OneHotEncoder` (inconnues ignorées) sur `nom_cols`, `OrdinalEncoder` avec les **deux** ordres sur `ord_cols`. `id`, `email` et `segment` ne sont pas des variables explicatives.

In [ ]:
from sklearn.compose import ColumnTransformer

ordres = [["Licence", "Master", "Doctorat"], ["faible", "moyenne", "elevee"]]

prep = ColumnTransformer([
    ("num", ..., num_cols),   # TODO
    ("nom", ..., nom_cols),   # TODO
    ("ord", ..., ord_cols),   # TODO
])

X_train = prep.fit_transform(df_train)   # fit sur le train
X_test = prep.transform(df_test)         # transform seulement sur le test
y_train, y_test = le.transform(df_train["segment"]), le.transform(df_test["segment"])

print(X_train.shape, X_test.shape)
pd.DataFrame(X_train, columns=prep.get_feature_names_out()).head()

✅ **Auto-correction**

In [ ]:
assert X_train.shape == (790, 12) and X_test.shape == (198, 12), "5 numériques + 5 villes + 2 ordinales = 12"
assert "email" not in " ".join(prep.get_feature_names_out()), "l'email n'est pas une variable explicative"
print("✅ 4.1 correct")

### 4.2 La normalisation change-t-elle le modèle ?
On prédit le `segment` avec un **k-NN** (k = 15), un modèle basé sur les distances. Comparez la **précision** (*accuracy*) sur le test :
- **sans normalisation** des variables numériques (`"passthrough"` à la place du `StandardScaler`) ;
- avec **Z-score** ;
- avec **Min-Max**.

💡 `Pipeline` enchaîne la préparation et le modèle : `fit` sur le train, `score` sur le test.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

def evaluer(normalisation):
    """Précision du k-NN (k=15) sur le test, pour une normalisation donnée des variables numériques."""
    prep = ColumnTransformer([
        ("num", normalisation, num_cols),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nom_cols),
        ("ord", OrdinalEncoder(categories=ordres), ord_cols),
    ])
    modele = Pipeline([("prep", prep), ("knn", KNeighborsClassifier(n_neighbors=15))])
    # TODO : entraîner sur (df_train, y_train), renvoyer le score sur (df_test, y_test)
    ...

resultats = pd.Series({
    "Aucune": ...,    # TODO : evaluer("passthrough")
    "Z-score": ...,   # TODO
    "Min-Max": ...,   # TODO
}, name="accuracy").round(3)
resultats

✅ **Auto-correction**

In [ ]:
assert resultats["Z-score"] > resultats["Aucune"] + 0.05, "la standardisation doit nettement améliorer le k-NN"
print("✅ 4.2 correct")

✍️ **Votre analyse :** expliquez l'écart entre « Aucune » et « Z-score » avec ce que vous avez vu en 2.1. Pourquoi Min-Max fait-il moins bien que Z-score ici ?

⭐ **Bonus :** ajoutez une transformation `log1p` du revenu avant le Z-score (indice : `FunctionTransformer(np.log1p)` et `make_pipeline`). Le résultat change-t-il ?

…

---
## Étape 5 — Documenter chaque décision
*Bonne pratique ⑥ : conserver le mapping utilisé, documenter pour la reproductibilité.*

✍️ **Journal des décisions** — une ligne par variable (ou groupe de variables).

| Variable(s) | Nature | Traitement retenu | Justification |
|---|---|---|---|
| `id`, `email` | … | … | … |
| `age`, `nb_enfants`, `anciennete`, `nb_achats` | … | … | … |
| `revenu` | … | … | … |
| `ville` | … | … | … |
| `niveau_etude`, `satisfaction` | … | … | … |
| `segment` | … | … | … |

✍️ **Conclusions** (5 à 10 lignes) : quelle normalisation, quels encodages, quel effet mesuré sur le modèle, quels pièges évités ?

…

---
## Bilan

| Besoin | Outil | À retenir |
|---|---|---|
| Même échelle, centré-réduit | `StandardScaler` | méthode par défaut, moins sensible aux outliers |
| Même échelle, borné [0, 1] | `MinMaxScaler` | très sensible aux valeurs extrêmes |
| Corriger l'asymétrie | `np.log1p` | change la **forme**, pas seulement l'échelle |
| Nominale (sans ordre) | `OneHotEncoder` / `get_dummies` | `handle_unknown="ignore"`, `drop_first` si modèle linéaire |
| Ordinale (avec ordre) | `OrdinalEncoder(categories=...)` | toujours donner l'ordre explicitement |
| Cible y | `LabelEncoder` | jamais pour les variables d'entrée X |
| Tout assembler | `ColumnTransformer` + `Pipeline` | `fit` sur le train, `transform` sur le test |

**Règle d'or :** le modèle ne doit jamais interpréter un ordre qui n'existe pas réellement.